In [3]:
import os
import pandas as pd
import re
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from dotenv import load_dotenv
from collections import Counter

class LattesNameExtractor:
    def __init__(self):
        # Configurações de ambiente e conexão
        load_dotenv(dotenv_path=r"E:\Python\.env")
        self.db_user = os.getenv('DB_USER')
        self.db_password = quote_plus(os.getenv('DB_PASSWORD'))
        self.db_host = os.getenv('DB_HOST')
        self.db_port = os.getenv('DB_PORT')
        self.db_name = os.getenv('DB_NAME')
        
        self.url_conexao = f"postgresql+psycopg2://{self.db_user}:{self.db_password}@{self.db_host}:{self.db_port}/{self.db_name}"
        self.engine = create_engine(self.url_conexao)
        
        # Estrutura de saída
        self.output_path = r"c:\temp"
        if not os.path.exists(self.output_path):
            os.makedirs(self.output_path)

    def fetch_names(self):
        """Executa consulta SQL otimizada usando a função customizada do ambiente."""
        query = """
            SELECT 
                jsonb_array_or_object_elements(json)->'CURRICULO-VITAE'->'DADOS-GERAIS'->>'@NOME-COMPLETO' as nome_completo
            FROM lattes_json
            WHERE json IS NOT NULL;
        """
        print("Extraindo nomes do banco de dados...")
        return pd.read_sql(query, self.engine)

    def process_dictionaries(self, df):
        """Processa a lista de nomes para gerar os dicionários de nomes, sobrenomes e stopwords."""
        nomes_set = set()
        sobrenomes_set = set()
        stopwords_set = set()
        
        # Regex para identificar stopwords (1-3 letras, minúsculas ou estrangeirismos comuns)
        # Inclui estrangeirismos conforme análise técnica
        re_stopword = re.compile(r'^([a-z]{1,3}|de|da|do|das|dos|e|la|del|von|van|di)$', re.IGNORECASE)

        for nome_completo in df['nome_completo'].dropna():
            partes = nome_completo.strip().split()
            if not partes:
                continue
            
            # Primeiro nome é sempre Nome
            nomes_set.add(partes[0])
            
            # Partes intermediárias e finais
            for i, parte in enumerate(partes[1:], 1):
                if re_stopword.match(parte):
                    stopwords_set.add(parte)
                elif i == len(partes) - 1:
                    # Última parte é quase sempre Sobrenome
                    sobrenomes_set.add(parte)
                else:
                    # Nomes do meio podem ser nomes ou sobrenomes; aqui tratamos como sobrenome
                    # para fins de anonimização (preservação do primeiro nome)
                    sobrenomes_set.add(parte)

        return sorted(list(nomes_set)), sorted(list(sobrenomes_set)), sorted(list(stopwords_set))

    def run(self):
        df_nomes = self.fetch_names()
        self.nomes, self.sobrenomes, self.stopwords = self.process_dictionaries(df_nomes)
        
        # Salvando os resultados
        pd.Series(self.nomes).to_csv(os.path.join("dic_nomes.csv"), index=False, header=['Nome'])
        pd.Series(self.sobrenomes).to_csv(os.path.join("dic_sobrenomes.csv"), index=False, header=['Sobrenome'])
        pd.Series(self.stopwords).to_csv(os.path.join("dic_stopwords.csv"), index=False, header=['Stopword'])
        
        print(f"Processamento concluído. Arquivos salvos em: {self.output_path}")
        print(f"Total de Nomes Únicos: {len(self.nomes)}")
        print(f"Total de Sobrenomes Únicos: {len(self.sobrenomes)}")
        print(f"Total de Stopwords: {len(self.stopwords)}")

if __name__ == "__main__":
    extractor = LattesNameExtractor()
    extractor.run()

Extraindo nomes do banco de dados...
Processamento concluído. Arquivos salvos em: c:\temp
Total de Nomes Únicos: 28621
Total de Sobrenomes Únicos: 75223
Total de Stopwords: 865
